In [10]:
# import the required packages
from openai import OpenAI
from sqlitesearch import TextSearchIndex
from dotenv import load_dotenv
import os
import json
from pprint import pprint

In [2]:
# load the environment variables
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [3]:
# connect to faq.db and check the number of documents
sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

In [4]:
# send a request to the LLM without the search function
openai_client = OpenAI(api_key=api_key)
messages = [
  {'role':'user', 'content':'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
  model='gpt-4o-mini',
  input=messages
)

print(response.output_text) # the LLM has no context of the query asked

To determine if you can join a course, you'll need to check the course's registration details. Here are some steps you can follow:

1. **Visit the Course Website:** Look for information about enrollment on the course's official webpage.
2. **Contact the Instructor or Administrator:** If the website doesn’t provide clear information, reach out directly to get details on availability and requirements.
3. **Check Deadlines:** Be aware of any registration deadlines that might affect your ability to join.
4. **Look for Prerequisites:** Ensure that you meet any prerequisites or criteria needed for enrollment.

If you provide more details about the specific course, I can offer more tailored advice!


In [5]:
# define the search function which searches through the persistent indexed documents
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
# define the search tool
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ dataset for the entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
# send the user query and include the tool in the request
response = openai_client.responses.create(
  model='gpt-4o-mini',
  input=messages,
  tools=[search_tool]
)

print(response.output) # the response contains a function call entry; LLM has asked to execute the 'search' function first

[ResponseFunctionToolCall(arguments='{"query":"join course"}', call_id='call_QT2Vtc9UdctFpihE9IC2OfyI', name='search', type='function_call', id='fc_088073e1cc01092e006a386a5e5258819cb904fd986e941868', namespace=None, status='completed')]


In [13]:
# parse the response object arguments
call = response.output[0]
args = json.loads(call.arguments)
# print(args) # {'query': 'joining the course'}
# call the search function with the arguments parsed from the function call
results = search(**args)
# convert the results in json format to append to 'messages'
results_json = json.dumps(results, indent=2)

In [ ]:
# append to messages a) add model conversation history (function call), and b) result of the function call
messages.extend(response.output)
# pprint(messages)
# [{'content': 'I just discovered the course. Can I join it?', 'role': 'user'},
#  ResponseFunctionToolCall(arguments='{"query":"join course"}', call_id='call_QT2Vtc9UdctFpihE9IC2OfyI', name='search', type='function_call', id='fc_088073e1cc01092e006a386a5e5258819cb904fd986e941868', namespace=None, status='completed')]
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json
})
# pprint(messages)

[{'content': 'I just discovered the course. Can I join it?', 'role': 'user'},
 ResponseFunctionToolCall(arguments='{"query":"join course"}', call_id='call_QT2Vtc9UdctFpihE9IC2OfyI', name='search', type='function_call', id='fc_088073e1cc01092e006a386a5e5258819cb904fd986e941868', namespace=None, status='completed'),
 {'call_id': 'call_QT2Vtc9UdctFpihE9IC2OfyI',
  'output': '[\n'
            '  {\n'
            '    "id": "74eb249bbf",\n'
            '    "course": "llm-zoomcamp",\n'
            '    "section": "General Course-Related Questions",\n'
            '    "question": "I just discovered the course. Can I still '
            'join?",\n'
            '    "answer": "Yes, but if you want to receive a certificate, you '
            'need to submit your project while we\\u2019re still accepting '
            'submissions."\n'
            '  },\n'
            '  {\n'
            '    "id": "bd31146b0e",\n'
            '    "course": "llm-zoomcamp",\n'
            '    "section": "Gener

In [17]:
# call the llm api with the expanded history
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

print(response.output_text)

Yes, you can still join the course! However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. You can start learning and submitting homework without formally registering.


In [18]:
# close the db connection
sqlite_index.close()